In [ ]:
import sys
sys.path.append('../../')
import os
import os.path as osp

import torch
from torch.utils.data import DataLoader
import numpy as np
from tqdm.notebook import tqdm

import qdre

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#DEVICE = 'cpu'
print(DEVICE)

/home/mdrnevich/test/QuasiDRE/SMEFT/../utils/__init__.py:4: UserWarning: The recommended fonts to use plothist were not found. You can install them by typing 'install_latin_modern_fonts' in your terminal. If it still does not work, please check the documentation at https://plothist.readthedocs.io/en/latest/usage/font_installation.html
  from . import plotting


cuda:0


In [2]:
DATA_DIR = "./data/"
SAVE = True

source_file = osp.join(DATA_DIR, "SMEFT_SM_combined_tuple")
target_file = osp.join(DATA_DIR, "SMEFT_EFT_combined_tuple")

if SAVE:
    for dataset in ["train", "val", "test"]:
        data_arrays_pos = []
        data_arrays_neg = []
        
        arr = np.load(source_file + "_" + dataset + ".npy")
        data_arrays_pos.append(arr[arr[:,-1] >= 0])
        data_arrays_neg.append(arr[arr[:,-1] < 0])

        arr = np.load(target_file + "_" + dataset + ".npy")
        data_arrays_pos.append(arr[arr[:,-1] >= 0])
        data_arrays_neg.append(arr[arr[:,-1] < 0])

        # Save the positively weighted datasets
        np.save(source_file + "_positives_" + dataset + ".npy", data_arrays_pos[0])
        np.save(target_file + "_positives_" + dataset + ".npy", data_arrays_pos[1])

        # Save the negatively weighted datasets
        np.save(source_file + "_negatives_" + dataset + ".npy", data_arrays_neg[0])
        np.save(target_file + "_negatives_" + dataset + ".npy", data_arrays_neg[1])

        del data_arrays_pos, data_arrays_neg

In [3]:
source_positive_file = source_file + "_positives"
source_negative_file = source_file + "_negatives"
target_positive_file = target_file + "_positives"
target_negative_file = target_file + "_negatives"

files = [source_positive_file, source_negative_file, target_positive_file, target_negative_file]
submodel_names = np.array(["Pos", "Neg", "Pos", "Neg"])

train_datasizes = []
val_datasizes = []
test_datasizes = []
for f in files:
    train_datasizes.append(np.load(f + "_train.npy").shape[0])
    val_datasizes.append(np.load(f + "_val.npy").shape[0])
    test_datasizes.append(np.load(f + "_test.npy").shape[0])

train_datasizes = np.array(train_datasizes)
val_datasizes = np.array(val_datasizes)
test_datasizes = np.array(test_datasizes)

train_datasizes, val_datasizes, test_datasizes

(array([776369,      0, 641157, 135212]),
 array([179162,      0, 148045,  31117]),
 array([238883,      0, 197103,  41780]))

### There are no negatives in the source dataset, so only use two subdensities

In [4]:
combos = [np.array((0,2)), #++
          np.array((0,3))] #+-

NUM_MODELS = 2

## Initiate the datasets

In [ ]:
training_settings = [{} for _ in range(NUM_MODELS)]

train_generator_datas = []
valid_generator_datas = []
test_generator_datas = []

for i in range(NUM_MODELS):
    sub_source_file = files[combos[i][0]]
    sub_target_file = files[combos[i][1]]
    min_datasizes = (train_datasizes[combos[i]].min(), val_datasizes[combos[i]].min(), test_datasizes[combos[i]].min())
        
    print(min_datasizes)
    training_settings[i].update({
        "source_file": sub_source_file,
        "target_file": sub_target_file,
        "n_train": int(min_datasizes[0]),
        "n_val": int(min_datasizes[1]),
        "n_test": int(min_datasizes[2])
    })
    print(sub_source_file, sub_target_file)


    train_base_dataset = qdre.preprocessing.Dataset(sub_source_file + "_train.npy", 0,
                                                     stop_event=training_settings[i]["n_train"])
    valid_base_dataset = qdre.preprocessing.Dataset(files[combos[i][0]] + "_val.npy", 0,
                                                     stop_event=training_settings[i]["n_val"])

    train_target_dataset = qdre.preprocessing.Dataset(sub_target_file + "_train.npy", 1,
                                                     stop_event=training_settings[i]["n_train"])
    valid_target_dataset = qdre.preprocessing.Dataset(files[combos[i][1]] + "_val.npy", 1,
                                                     stop_event=training_settings[i]["n_val"])


    source_weight_norm = train_base_dataset.process(normalize_weights=True)
    source_valid_weight_norm = valid_base_dataset.process(normalize_weights=True)
    
    target_weight_norm = train_target_dataset.process(normalize_weights=True)
    target_valid_weight_norm = valid_target_dataset.process(normalize_weights=True)

    train_generator_datas.append(qdre.preprocessing.CombinedDataset(train_base_dataset, train_target_dataset))
    valid_generator_datas.append(qdre.preprocessing.CombinedDataset(valid_base_dataset, valid_target_dataset))

(np.int64(641157), np.int64(148045), np.int64(197103))
./data/SMEFT_SM_combined_tuple_positives ./data/SMEFT_EFT_combined_tuple_positives
(np.int64(135212), np.int64(31117), np.int64(41780))
./data/SMEFT_SM_combined_tuple_positives ./data/SMEFT_EFT_combined_tuple_negatives


## Do some data preprocessing for standardized inputs and weights

In [ ]:
X_scalers, train_weight_norms = list(zip(*[qdre.preprocessing.get_scaling(train_generator_data) for train_generator_data in train_generator_datas]))
_, valid_weight_norms = list(zip(*[qdre.preprocessing.get_scaling(valid_generator_data) for valid_generator_data in valid_generator_datas]))
print(train_weight_norms, valid_weight_norms)

100%|██████████| 61/61 [00:00<00:00, 185.05it/s]

(tensor(1.0000), tensor(1.0000)) (tensor(1.0000), tensor(1.0000))


## Prepare the data for training

In [7]:
random_seed = 0

torch.manual_seed(random_seed)
batch_sizes = [int(2**8), int(2**8)]
[ts.update({
    "batch_size": batch_sizes[i],
    "random_seed": random_seed
}) for i, ts in enumerate(training_settings)]

train_loaders = [DataLoader(train_generator_data, batch_size=batch_sizes[i], shuffle=True) for i, train_generator_data in enumerate(train_generator_datas)]
valid_loaders = [DataLoader(valid_generator_data, batch_size=batch_sizes[i], shuffle=False) for i, valid_generator_data in enumerate(valid_generator_datas)]

## Construct the model

In [ ]:
inputs = 16
hidden_nodes = [128, 128, 128]
outputs = 1

models = [qdre.models.Classifier(inputs, hidden_nodes, outputs).to(DEVICE) for _ in range(NUM_MODELS)]
print(models)
print("----------")

#model = model.to(DEVICE)

learning_rate = 1e-3
optimizers = [torch.optim.Adam(model.parameters(), lr=learning_rate) for model in models]

[ts.update({
    "learning_rate": learning_rate,
    "optimizer": "Adam"
}) for ts in training_settings]

[Classifier(
  (model): Sequential(
    (0): Linear(in_features=16, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=1, bias=True)
    (7): Sigmoid()
  )
), Classifier(
  (model): Sequential(
    (0): Linear(in_features=16, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=1, bias=True)
    (7): Sigmoid()
  )
)]
----------


[None, None]

In [9]:
SAVE_DIR = "./models/"
if not osp.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

## Perform the training

In [ ]:
for i in range(NUM_MODELS):
    modname = "classifier_subdensity_{}_batch{}".format(''.join(submodel_names[combos[i]]), batch_sizes[i])
    model = models[i]
    optimizer = optimizers[i]
    X_scaler = X_scalers[i]
    train_weight_norm = train_weight_norms[i]
    valid_weight_norm = valid_weight_norms[i]
    train_loader = train_loaders[i]
    valid_loader = valid_loaders[i]
    lossfunction = 'bce' #'bce', 'sqrt'
    regularization = 0 #1e-5 #0.01
    
    n_epochs = 1500
    stale_epochs = 0
    best_valid_loss = 99999
    patience = [15, 15]
    max_num_batches = int(int(2e4) / batch_sizes[i])
    t = tqdm(range(0, n_epochs))
    
    training_losses = [qdre.train.test(
            model,
            train_loader,
            X_scaler,
            weight_norm=train_weight_norm,
            max_num_batches=max_num_batches,
            loss=lossfunction,
            regularization=regularization,
            device=DEVICE,
            progress_bar=False,
            leave=False
        )[0],]
    validation_losses = [qdre.train.test(
            model,
            valid_loader,
            X_scaler,
            weight_norm=valid_weight_norm,
            loss=lossfunction,
            regularization=regularization,
            device=DEVICE,
            progress_bar=False,
            leave=False
        )[0],]
    
    
    for epoch in t:
        loss = qdre.train.train(
            model,
            optimizer,
            train_loader,
            X_scaler,
            weight_norm=train_weight_norm,
            max_num_batches=max_num_batches,
            loss=lossfunction,
            regularization=regularization,
            device=DEVICE,
            progress_bar=False,
            leave=bool(epoch == n_epochs - 1),
        )
        #loss -= optimal_train_loss
        training_losses.append(loss[0])
    
        valid_loss = qdre.train.test(
            model,
            valid_loader,
            X_scaler,
            weight_norm=valid_weight_norm,
            loss=lossfunction,
            regularization=regularization,
            device=DEVICE,
            progress_bar=False,
            leave=bool(epoch == n_epochs - 1),
        )
        #valid_loss -= optimal_valid_loss
        validation_losses.append(valid_loss[0])
        print("Epoch: {:02d}, Training Loss:   {:.4f}".format(epoch, loss[1]))
        print("           Validation Loss: {:.4f}".format(valid_loss[1]))
    
        if valid_loss[1] < best_valid_loss:
            best_valid_loss = valid_loss[1]
            training_settings[i].update({
                "n_epochs": epoch+1,
                "training_losses": training_losses,
                "validation_losses": validation_losses
            })
            model_metadata = qdre.train.get_model_metadata(training_settings[i], model, X_scaler, train_weight_norm)
            qdre.train.save_model_data(model, model_metadata, savedir=SAVE_DIR, name=modname, save_onnx=False, device=DEVICE)
            print("New best model saved to: {}.zip".format(osp.join(SAVE_DIR, modname)))
            stale_epochs = 0
        else:
            print("Stale epoch")
            stale_epochs += 1
        if stale_epochs >= patience[i]:
            print("Early stopping after %i stale epochs" % patience[i])
            break

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch: 00, Training Loss:   0.6438
           Validation Loss: 0.6406
New best model saved to: ./models/classifier_subdensity_PosPos_batch256.zip
Epoch: 01, Training Loss:   0.6271
           Validation Loss: 0.6235
New best model saved to: ./models/classifier_subdensity_PosPos_batch256.zip
Epoch: 02, Training Loss:   0.6138
           Validation Loss: 0.6187
New best model saved to: ./models/classifier_subdensity_PosPos_batch256.zip
Epoch: 03, Training Loss:   0.6085
           Validation Loss: 0.6132
New best model saved to: ./models/classifier_subdensity_PosPos_batch256.zip
Epoch: 04, Training Loss:   0.6026
           Validation Loss: 0.6101
New best model saved to: ./models/classifier_subdensity_PosPos_batch256.zip
Epoch: 05, Training Loss:   0.6026
           Validation Loss: 0.6057
New best model saved to: ./models/classifier_subdensity_PosPos_batch256.zip
Epoch: 06, Training Loss:   0.6004
           Validation Loss: 0.6035
New best model saved to: ./models/classifier_subdensit

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch: 00, Training Loss:   0.4626
           Validation Loss: 0.3863
New best model saved to: ./models/classifier_subdensity_PosNeg_batch256.zip
Epoch: 01, Training Loss:   0.3519
           Validation Loss: 0.3277
New best model saved to: ./models/classifier_subdensity_PosNeg_batch256.zip
Epoch: 02, Training Loss:   0.3166
           Validation Loss: 0.2956
New best model saved to: ./models/classifier_subdensity_PosNeg_batch256.zip
Epoch: 03, Training Loss:   0.2865
           Validation Loss: 0.2826
New best model saved to: ./models/classifier_subdensity_PosNeg_batch256.zip
Epoch: 04, Training Loss:   0.2838
           Validation Loss: 0.2730
New best model saved to: ./models/classifier_subdensity_PosNeg_batch256.zip
Epoch: 05, Training Loss:   0.2668
           Validation Loss: 0.2712
New best model saved to: ./models/classifier_subdensity_PosNeg_batch256.zip
Epoch: 06, Training Loss:   0.2683
           Validation Loss: 0.2631
New best model saved to: ./models/classifier_subdensit